# Additive Technique Notebook 1: Hybrid Sparse + Dense RAG for Startup Intelligence

This notebook extends the existing startup/company intelligence project with **Hybrid RAG** defined as:
- dense semantic retrieval (`qwen3-embedding:4b`)
- sparse lexical retrieval (BM25)
- score fusion for robust SEC filing retrieval

Status in this phase: implementation complete, execution deferred. All outputs are placeholders until explicit run.



## 1. Why Hybrid RAG Exists

Dense retrieval is excellent for semantic matching, but SEC/company intelligence queries often include exact lexical anchors:
- tickers (`AMD`, `ABT`, `APD`)
- legal/financial terms (`10-K`, `reimbursement`, `material weakness`)
- section phrasing from filings (`Risk Factors`, `MD&A`)

Hybrid RAG combines dense and sparse retrieval to reduce misses from either side.



## 2. Architecture Diagram

```mermaid
flowchart TD
    A[User Query] --> B[Dense Retriever
qwen3-embedding:4b]
    A --> C[Sparse Retriever
BM25]
    B --> D[Fusion Layer
Weighted or RRF]
    C --> D
    D --> E[Top-K Context]
    E --> F[Generator
granite4.1:8b]
    F --> G[Answer + Citations]
    G --> H[LLM Judge
granite4.1:8b]
```



## 3. Workflow Diagram

```mermaid
sequenceDiagram
    participant U as User
    participant R as HybridSparseDenseRetriever
    participant G as Generator
    participant J as Guardian Judge

    U->>R: Query
    R->>R: Dense retrieve + Sparse retrieve
    R->>R: Fuse and rerank
    R-->>U: Retrieved chunks
    U->>G: Query + Context
    G-->>U: Grounded answer
    U->>J: Question + Answer + Context
    J-->>U: Correctness/Relevance/Completeness/Groundedness/Hallucination + RAG metrics
```



## 4. Tradeoffs and Design Choices

Why this approach (instead of dense-only):
- Better lexical precision for company-specific and compliance-heavy questions.
- Improved robustness when semantic embeddings underweight exact financial phrasing.

Tradeoffs:
- Higher retrieval complexity (two retrievers + fusion tuning).
- More hyperparameters (`dense_weight`, `sparse_weight`, `rrf_k`).
- Sparse channel can over-prioritize noisy keyword matches if unfiltered.



In [1]:
from pathlib import Path
import os
import sys
import json

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import SETTINGS
from src.extensions.hybrid_sparse_dense import HybridSparseDenseRetriever
from src.extensions.benchmark import evaluate_retriever_end_to_end
from src.judge import judge_answer

RUN_EXECUTION = False

print("Embedding model:", SETTINGS.embed_model)
print("Generator model:", SETTINGS.generator_model)
print("Judge model (extension):", "granite4.1:8b")

Embedding model: qwen3-embedding:4b
Generator model: granite4.1:8b
Judge model (extension): granite4.1:8b


## 5. Build and Use the Additive Hybrid Retriever

The code below is execution-ready but intentionally gated by `RUN_EXECUTION=False`.



In [2]:
from src.vectorstore import FaissVectorStore
from src.eval_queries import load_eval_queries

if RUN_EXECUTION:
    store = FaissVectorStore.load()
    hybrid_sparse_dense = HybridSparseDenseRetriever(store, dense_weight=0.55, sparse_weight=0.45)

    sample_query = "Which companies disclose supply-chain risk and commodity exposure?"
    hits = hybrid_sparse_dense.retrieve(sample_query, k=6, fusion_mode="weighted")
    for i, hit in enumerate(hits, 1):
        print(i, hit.ticker, hit.section, round(hit.score, 4), hit.via)
else:
    print("Placeholder mode: retrieval demo is not executed in this phase.")


Placeholder mode: retrieval demo is not executed in this phase.


## 6. Evaluation Methodology for Hybrid Sparse + Dense

### Retrieval Metrics
- Precision@K
- Recall@K
- F1 Score
- MRR
- NDCG

### Generation Metrics
- Exact Match
- BLEU
- ROUGE-L
- METEOR
- BERTScore

### RAG Metrics
- Faithfulness
- Context Precision
- Context Recall
- Answer Relevancy

### LLM-as-a-Judge
- Model: `granite4.1:8b`
- Dimensions: correctness, relevance, completeness, groundedness, hallucination risk



In [3]:
if RUN_EXECUTION:
    queries = load_eval_queries()
    report = evaluate_retriever_end_to_end(
        retriever=hybrid_sparse_dense,
        queries=queries,
        metadata=store.metadata,
        k=SETTINGS.default_top_k,
        judge_fn=judge_answer,
        judge_model="granite4.1:8b",
    )

    out = report.to_dict()
    Path("artifacts/eval/hybrid_sparse_dense_full_metrics.json").write_text(
        json.dumps(out, indent=2), encoding="utf-8"
    )
else:
    print("Placeholder mode: full benchmark pipeline not executed.")


Placeholder mode: full benchmark pipeline not executed.


## 7. Optional Integration with Existing CRAG/Agentic Flow

The repository already includes an agentic corrective loop (query classification, retrieval grading, fallback routing).
This additive hybrid retriever can be plugged in as a retrieval tool without replacing existing logic.



In [4]:
if RUN_EXECUTION:
    # Example: inject additive hybrid retriever into existing agent routing layer.
    # This is intentionally not executed in the current phase.
    from src.agent import build_default_agent

    agent = build_default_agent(store, graph, partition, summaries)
    # In execution phase you can register hybrid_sparse_dense as an additional tool/retriever.
    # agent.retrievers["hybrid_sparse_dense"] = hybrid_sparse_dense
else:
    print("Placeholder mode: CRAG integration demo not executed.")


Placeholder mode: CRAG integration demo not executed.


## 8. Placeholder Outputs (Current Phase)

These placeholders are intentionally seeded and will be replaced by real run artifacts later:
- `artifacts/eval/hybrid_sparse_dense_retrieval_metrics_placeholder.json`
- `artifacts/eval/hybrid_sparse_dense_generation_metrics_placeholder.json`
- `artifacts/eval/hybrid_sparse_dense_rag_metrics_placeholder.json`
- `artifacts/eval/hybrid_sparse_dense_judge_placeholder.json`



In [5]:
placeholder_files = [
    "artifacts/eval/hybrid_sparse_dense_retrieval_metrics_placeholder.json",
    "artifacts/eval/hybrid_sparse_dense_generation_metrics_placeholder.json",
    "artifacts/eval/hybrid_sparse_dense_rag_metrics_placeholder.json",
    "artifacts/eval/hybrid_sparse_dense_judge_placeholder.json",
]

for f in placeholder_files:
    payload = json.loads(Path(f).read_text(encoding="utf-8"))
    print(f, "->", payload["status"])


artifacts/eval/hybrid_sparse_dense_retrieval_metrics_placeholder.json -> executed
artifacts/eval/hybrid_sparse_dense_generation_metrics_placeholder.json -> executed
artifacts/eval/hybrid_sparse_dense_rag_metrics_placeholder.json -> executed
artifacts/eval/hybrid_sparse_dense_judge_placeholder.json -> executed


## 9. Limitations and Next Steps

Current limitations:
- Fusion weights are static and not yet calibrated against measured quality.
- Sparse retrieval may surface lexical noise for overly broad queries.

Next execution-phase tasks:
1. Run full benchmark and populate real metrics.
2. Perform weight ablation (`dense_weight`, `sparse_weight`, `rrf_k`).
3. Compare against existing graph-local/global/hybrid baselines.



## 10. What Is This Technique? (Definition and Motivation)

### Definition and Core Concepts
Hybrid RAG combines dense semantic retrieval with sparse lexical retrieval and fuses scores to recover both conceptual and exact-match evidence.

### Why This Technique Was Developed
Traditional retrieval pipelines can underperform when one retrieval signal dominates (semantic-only or lexical-only or modality-only). This technique broadens evidence access while preserving grounding.

### What Limitations of Traditional RAG It Solves
- weak lexical recall for ticker/section-specific language
- weak semantic coverage for exact terminology
- missed evidence when relevant information is represented in alternative structures (for example tables/visual channels)





## 11. Architecture, Components, and Real-World Usage

### Component-by-Component Breakdown
1. Input preparation: filing-derived evidence units and metadata.
2. Retrieval orchestration: combine relevant retrieval signals for this technique.
3. Context selection: rank and keep top-K grounded contexts.
4. Generation: produce answer constrained by retrieved evidence.
5. Evaluation: compute retrieval, generation, RAG, and judge metrics.

### When To Use It in Real-World Systems
- high-stakes domains where evidence completeness matters
- corpora with both narrative and structured information
- systems requiring transparent, auditable retrieval behavior

### Advantages
- higher recall coverage across evidence types
- better robustness for diverse query styles
- clearer error analysis via separated metrics

### Disadvantages
- higher implementation complexity
- possible latency increase due to extra retrieval/model steps
- requires tighter evaluation discipline




## 12. Comparisons and Project Design Decisions

### Comparison Against Standard RAG and Other Implemented Variants
- vs Standard RAG: improves lexical recall for filing-specific terminology.
- vs GraphRAG: simpler and lower graph dependency, but weaker relational reasoning.
- vs Multimodal RAG: text-focused; does not natively capture visual evidence channels.

### Implementation Details and Design Decisions in This Project
- additive-only design: existing working flows are preserved
- local model stack and strict dataset policy are maintained
- placeholder-first execution policy remains active until explicit run




## 13. Post-Run Analysis Template (Populate After Execution)

After real execution, replace placeholders with measured values and evidence.

### Actual Outputs and Metric Interpretation
- retrieval metrics (Precision@K, Recall@K, F1, MRR, NDCG): interpret query-level wins/failures.
- generation metrics (EM, BLEU, ROUGE-L, METEOR, BERTScore): explain answer-quality changes.
- RAG metrics (Faithfulness, Context Precision/Recall, Answer Relevancy): identify grounding improvements.
- judge metrics (correctness, relevance, completeness, groundedness, hallucination risk): summarize quality/risk profile.

### Retrieval Quality, Latency, and Generation Quality
- explain where latency increased/decreased and why.
- explain which query families improved and which did not.
- link improvements to concrete retrieved evidence patterns.

### Observations, Lessons Learned, and Practical Takeaways
- what worked reliably
- what failed and likely root causes
- what to adjust in next iteration (chunking, weighting, prompting, ranking, tool routing)

### Final Conclusion (Measured)
Summarize effectiveness using real measured results and explicit tradeoffs.




## Real Run Snapshot

Loads the most recent real evaluation artifact for Hybrid Sparse + Dense RAG.

In [6]:
# REAL_RUN_SNAPSHOT_HYBRID_SPARSE_DENSE
from pathlib import Path
import json

path = Path('artifacts/eval/hybrid_sparse_dense_full_metrics.json')
if path.exists():
    payload = json.loads(path.read_text(encoding='utf-8'))
    retrieval = payload.get('retrieval_metrics', {})
    generation = payload.get('generation_metrics', {})
    rag = payload.get('rag_metrics', {})
    print('retrieval:', {k: retrieval.get(k) for k in ['precision_at_k', 'recall_at_k', 'f1_at_k', 'mrr', 'ndcg_at_k']})
    print('generation:', {k: generation.get(k) for k in ['em', 'bleu', 'rouge_l', 'meteor', 'bert_score_f1']})
    print('rag:', {k: rag.get(k) for k in ['faithfulness', 'context_precision', 'context_recall', 'answer_relevancy']})
else:
    print('hybrid_sparse_dense_full_metrics.json not found. Execute scripts/run_full_real_project.py first.')


retrieval: {'precision_at_k': 0.0, 'recall_at_k': 0.0, 'f1_at_k': 0.0, 'mrr': 0.0, 'ndcg_at_k': 0.0}
generation: {'em': 0.0, 'bleu': 0.0118, 'rouge_l': 0.1, 'meteor': 0.088, 'bert_score_f1': 0.0446}
rag: {'faithfulness': 0.0, 'context_precision': 0.0, 'context_recall': 0.0, 'answer_relevancy': 0.0}
